In [ ]:

from datetime import datetime, timezone
import time
from oandapyV20 import API
import logging

import config
import oanda_api_handler
import EMA_Trend_following as strategy

In [ ]:
def run_bot():
    print("Starting pyTrade Bot ......")

    #Configure Logging
    logging.basicConfig(
        filename="v20.log",
        level=logging.INFO,
        format='%(asctime)s [%(levelname)s] %(name)s : %(message)s',
    )


    client =  API(access_token=config.OANDA_API_KEY, environment=config.OANDA_ENVIRON)
    # Turn connected indicator green. Add some logic to check client is valid.
    last_checked = None

    while True: # Change to while GUI start/stop toggle, toggled to start
        current_time = datetime.now(timezone.utc)
        # Check for new 15 min candle
        if current_time.minute % 15 == 0 and current_time.second < 10:
            # Check if the 15 min interval has changed since the last check
            if last_checked != current_time.minute:
                print (f"Current Time: {current_time}")
                print("Checking for trade signals")

                price = oanda_api_handler.get_Candles(client)
                if price is not None:
                    price = strategy.get_indicators(price, high_ind=50, low_ind=20)
                else:
                    print("Failed to get Candles")
                
                order = strategy.strat_5020_Pullback(price)
                if order is None:
                    print("Strategy conditions not met")
                else:
                    print("Trade detected & order placed:\n"
                    f'Instrument: {order["order"]["instrument"]}\n'
                    f'Units: {order["order"]["units"]}\n'
                    f'Stop Loss: {order["order"]["stopLossOnFill"]}\n'
                    f'Take Profit: {order["order"]["takeProfitOnFill"]}\n')
                    # Requires some error handling to confirm order placed and
                    # check there are no open orders........
                    oanda_api_handler.place_order(order)
                    # Wait for order to close before continueing.
                last_checked = current_time.minute # Update to prevent re-triggering

        time.sleep(10) 


In [ ]:
run_bot()
